In [ ]:
from datetime import datetime
from opera_utils import disp
import geopandas as gpd
import os
from pathlib import Path
import subprocess
from transboundary_opera import displacement_tools as dt
import matplotlib.pyplot as plt
import xarray as xr
from shapely.geometry import mapping
import ultraplot as uplt
from pyproj import CRS
import asf_search as asf
import re

# %matplotlib widget
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# TODO: Maybe look at making this a pixi task? Could make it easier to run?

In [ ]:
SEARCH_START = datetime(2024, 6, 1)     # next runs should start at Jan 1, 2016
SEARCH_END = datetime(2025, 1, 1)

gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")
# frame_ids = dt.get_opera_frame_ids(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")     # this only gets one set of frame ids 

In [ ]:
# Initialize a list to store frame IDs for each row
all_frame_ids = []

for entry, row in gdf.iterrows():
    
    results = asf.geo_search(
        intersectsWith=row.geometry.convex_hull.wkt,
        dataset=asf.DATASET.OPERA_S1,
        processingLevel=asf.PRODUCT_TYPE.DISP_S1,
        maxResults=50  # We only need a few results to identify the Frame IDs
    )
    
    # Set for the current row
    current_frame_ids = set()
    pattern = re.compile(r'_F(\d{5})_')

    for product in results:
        filename = product.properties['fileName']
        match = pattern.search(filename)
        if match:
            frame_id_str = match.group(1) 
            current_frame_ids.add(int(frame_id_str))
    
    # Add the sorted list of frame IDs to our collection
    all_frame_ids.append(sorted(list(current_frame_ids)))

# Assign the collected frame IDs to a new column in the GeoDataFrame
gdf['frame_ids'] = all_frame_ids

unique_frame_ids = sorted({fid for ids in gdf['frame_ids'] if isinstance(ids, (list, tuple, set)) for fid in ids})
len(unique_frame_ids)

In [ ]:
# path to mintpy bash script
script_path = Path(os.getcwd()).parent / 'create-mintpy_cc.sh'

for frame_id in unique_frame_ids:
    out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame_id)
    os.makedirs(out_path, exist_ok=True)

    if (out_path / "mintpy").exists():
        print(f"Frame ID {frame_id} already processed. Skipping.")
        continue
    
    else:
        # 2. Define the REQUIRED environment variables for the script
        required_env = {
            "FRAME_ID": f"{frame_id}",  # Using the current frame ID from the loop
            "START": SEARCH_START.strftime("%Y-%m-%d"),
            "END": SEARCH_END.strftime("%Y-%m-%d"),
            }

        # 3. Define OPTIONAL environment variables to override the script's defaults
        optional_env = {
            "NUM_WORKERS": "8",              # Overrides default of 4
            "REFERENCE_METHOD": "HIGH_COHERENCE"    # Overrides default of BORDER. Either HIGH_COHERENCE or BORDER would be best I think.
        }
        print(f"Preparing to execute script: {script_path}")
        print(f"With required environment: {required_env}")

        # Construct the full environment: current system environment + custom variables
        full_environment = os.environ.copy()
        full_environment.update(required_env)
        full_environment.update(optional_env)

        try:
            # Execute the script directly (preferred over shell=True)
            result = subprocess.run(
                [script_path],
                check=True,  # Raise CalledProcessError for non-zero exit codes
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,  # Decode output as string
                env=full_environment,
                cwd = out_path
            )

            # Success output
            print("\nScript Execution Successful")
            print(f"Return Code: {result.returncode}")
            print("\nSTDOUT")
            print(result.stdout)
            if result.stderr:
                print("\nSTDERR (Non-fatal messages)")
                print(result.stderr)
        except subprocess.CalledProcessError as e:
            # Error handling for script failure (e.g., if a command inside the script fails)
            print(f"\nERROR: Script Failed (Exit Code {e.returncode})")
            print(f"Command run: {e.cmd}")
            print("\nSTDOUT (Captured before failure)")
            print(e.stdout)
            print("\nSTDERR (Error messages)")
            print(e.stderr)

        except FileNotFoundError:
            # Error handling for when the script file itself cannot be found
            print(f"\nCRITICAL ERROR: Script file not found or lacks execute permission at: {script_path}")